In [8]:
!pip install psutil

In [9]:
# Cell 2
import time
import psutil
import os
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings

warnings.filterwarnings('ignore')

# Import your polished engine!
from tisd_engine import chat_with_tisd

print("✅ Telemetry Engine and TISD Model Loaded.")

✅ Telemetry Engine and TISD Model Loaded.


In [10]:
# Cell 3
# UI Elements
title = widgets.HTML("<h2>🔬 TISD: Engineering Telemetry Dashboard</h2>")
text_input = widgets.Text(placeholder='Ask a custom question...', description='Query:', layout=widgets.Layout(width='70%'))
class_dropdown = widgets.Dropdown(options=[('All Classes', None), ('Class 1', '1'), ('Class 2', '2'), ('Class 3', '3'), ('Class 4', '4')], description='Filter:')
button = widgets.Button(description='Run Inference', button_style='danger')
output_area = widgets.Output()

# Process monitor
process = psutil.Process(os.getpid())

def run_telemetry(b):
    with output_area:
        output_area.clear_output()
        question = text_input.value
        class_filter = class_dropdown.value
        
        if not question.strip():
            print("Please enter a query.")
            return
            
        print("⚡ Warming up Apple Silicon MPS...")
        
        # --- SNAPSHOT BEFORE ---
        ram_before = process.memory_info().rss / (1024 * 1024) # in MB
        psutil.cpu_percent(interval=None) # Prime the CPU monitor
        
        # --- INFERENCE ---
        ans, ctx, latency = chat_with_tisd(question, class_filter=class_filter)
        
        # --- SNAPSHOT AFTER ---
        ram_after = process.memory_info().rss / (1024 * 1024) # in MB
        cpu_spike = psutil.cpu_percent(interval=None)
        ram_diff = ram_after - ram_before
        
        output_area.clear_output()
        
        # --- RENDER RESULTS ---
        display(HTML(f"<h3>🧑‍🎓 Query: {question}</h3>"))
        display(HTML(f"<div style='background-color:#e8f5e9; padding:15px; border-left: 5px solid #4CAF50; border-radius: 5px; font-size: 16px;'>"
                     f"<b>🤖 TISD Output:</b><br>{ans}</div>"))
        
        # --- RENDER TELEMETRY ---
        telemetry_html = f"""
        <div style='background-color:#263238; color:#00E676; padding:15px; border-radius: 5px; margin-top:20px; font-family: monospace;'>
            <b>📊 M4 COMPUTE TELEMETRY</b><br>
            ----------------------------------------<br>
            ⏱️ <b>Total Latency:</b>     {latency:.3f} seconds<br>
            🧠 <b>RAM Allocated:</b>     {ram_after:.2f} MB (Delta: {ram_diff:+.2f} MB)<br>
            ⚙️ <b>Process CPU Load:</b>  {cpu_spike}% (Includes MPS offloading)<br>
            📚 <b>Context Chunks:</b>    Retrieved {len(ctx.split())} words<br>
            ----------------------------------------<br>
            <i>Note: For exact Wattage, run 'sudo powermetrics' in terminal.</i>
        </div>
        """
        display(HTML(telemetry_html))
        
        # FIXED: Expandable Context (No 'with' statement)
        acc = widgets.Accordion(children=[widgets.HTML(f"<p>{ctx}</p>")])
        acc.set_title(0, '🔍 View Retrieved Context')
        display(acc)

button.on_click(run_telemetry)

# Display the dashboard
display(widgets.VBox([title, widgets.HBox([text_input, class_dropdown, button]), output_area]))